Q.1:
Architecture

3. On-Prem Architecture
Propose a conceptual architecture (tools and basic steps) for deploying the pipeline and model into production on the company's On-Premise infrastructure. How would you handle the regular updating (retraining) of the model?


A.1: 

* If i take into consideration that churn model downstream intervention is almost always planned intervention (retention mail, bonus, etc...), none of these are the ones that needs to happen "instant".
* Signals on which churn are calculated are also inside given window in my case ~90 days therefore this is also hard limitation that points toward non latency. 
Based on this i would focus on serving patterns as **batch scoring**.

Pipeline stages:
|Stage|Tools|Purpose|
|-----|-----|-------|
|1. Input| Airflow (scheduled ETL)| Pull from the betting platform transactional DB into on-prem warehouse (Postgres)|
|2. Feature engineering | Spark/pandas + Feast feature store | Build the RFM + engineered features for model training. Feast solution makes training and serving read identical definitions which helps in avoiding train/serve skew.|
|3. Training and Tracking| MLflow (tracking + model registry)| Log experiments (PR-AUC, ROC-AUC etc...) Also can register models with stages in production.|
|4. Packing| Docker| Model + inference code as versioned image.|
|5. Serving| Batch Airflow job or FastAPI| Score all players weekly/monthly and write a table. THen add a REST endpoint only if on demand scoring is needed|
|6. CI/CD| On-prem Kubernetes + GitLab CI| Scale batch jobs/serving containers. Automate test -> build -> deploy.|
|7. Monitoring| Evidently AI + Grafana| Feature/prediction drift (PSI), serving health, alert on rolling holdout AUC/PR-AUC decay.|

* **Retraining:**

    **1. Schedule retrain:** timing should be tied to the observation window (monthly, 90 days etc...) so that the model can track recent player behaviour.

    **2. Drift-triggered retrain**: supplement the schedule with triggers on feature drift (PSI: Population Stability Index) and performance decay (rolling PR-AUC / lift@top-5% on a fresh holdout). I pair these two because they catch different failure modes at different times: PSI compares the current feature (or score) distribution against the training baseline and can flag a shift immediately, without needing ground truth labels. 
    PR-AUC/lift decay needs churn labels, which only arrive weeks later, so it's a lagging signal. PSI acts as an early warning while PR-AUC/lift confirms whether that shift actually hurt the model.

    **3. Validation of the models:** Auto promotion is out of the picture, newly trained model must beat the current production model on set of data that is excluded from training (PR-AUC, calibration, business metrics, etc...) before it can be cleared for promotion in the MLflow registry.

    **4. Data and model versioning:** snapshot each runs training data (DVC, or timestamped Parquet as a lighter alternative) alongside the corresponding MLflow model version. This gives full reproducibility and an audit trail which is important in a regulated betting context, where we may need to reconstruct exactly on what data a specific model version was trained on.

    **5. Human-in-the-loop gate:** promotion from Staging to Production isnt automatic. Someone reviews the new model metrics against the current best one and confirms before it goes live. This way a model that looks good on paper but has some subtle issue cant quietly start driving CRM decisions on its own.

![On-Prem Churn Model Architecture](../Data/Diagrams/123456.drawio.png)

Q.2:

4. Utilizing Text Data from the Contact Center
In addition to the structured data from the CSV, the company also possesses the history of text correspondences that dissatisfied users had with our contact center. How would you leverage this text data to improve the existing Churn system? Conceptually describe how you see the application of AI / RAG systems in this scenario and what the technical steps for implementation would be.


A.2:

* The structured CSV tells us **what** a player did (declining bets, rising days since last activity, shrinking deposits). The contact center text tells us **why** (a withdrawal delay, bonus dissatisfaction, an escalating tone) and that "why" is often a leading indicator that appears before the behavioural drop the current model relies on.

I see two complementary applications. First, use the contact center data to create new structured features for the churn model. Second, build a RAG assistant that helps retention agents understand why a player is at risk and recommend appropriate actions.


Basically we would have two parts:
* **Application 1, Text -> Feature (offline, feeds the existing churn model):**
    * This would be LLM extraction step. For each correspondence i run one structured call that emits a fixed schema (complaint_category[withdrawal/bonus/odds...], escalation_flag, resolution_status, etc...) with deterministic settings (temperature = 0, fixed seeds, etc) so that the features can be reproducible.
        
    * Then i would aggregate to the player/observation_window level and jow new columns onto the feature table (e,g. complain_count_30d, has_withdrawal_complaint, etc...) that was used to train churn model. These would drop straight into the existing pipeline and can be evaluated the same way (same metrics), so that in the end i can qunatify their marginal lift with the separate table already in the notebook.

* **Application 2, RAG decision support (online, agent-facing):**
    * When a retention agent opens a player and model flagged it as high-risk, they see a grounded summary of that players own contact history next to the churn score. Now they can ask "what has this player complained about?" or "what happened to this player?". The answer is generated then only from that players retrieved messages, with citations so it can be auditable.
    This supports the retention playbook knowledge base to suggest a grounded next best action.

* **Technical implementation**

| Stage | Tools | Purpose |
|-------|-------|---------|
| 1. Ingestion & PII redaction | ETL scripts + regex/pattern matching | Collect transcripts/emails/chat. Redact PII (names, IBAN, card numbers, doc IDs)|
| 2. Structured extraction | Azure OpenAI / Open-weight LLM (deterministic, `temperature=0`) | Per contact structured call -> fixed schema (sentiment, complaint category, escalation, competitor mention, etc...) |
| 3. Aggregate & join | pandas / Spark | Roll up to player/window level. Merge as new features into table for model training (complaint_count_30d, avg_sentiment_30d, etc...) |
| 4. Chunk + embed + index | LangChain RecursiveCharacterTextSplitter + self hosted embedding model + Milvus/Qdrant/other | Chunk transcripts, embed with self hosted model, upsert into vector DB with player_id + timestamp metadata |
| 5. Retrieval service | Vector DB (Milvus / Qdrant / etc...) | Filter by player_id -> top-k ANN search -> recency/relevance filtering |
| 6. Grounded generation | LLM (hybrid or self-hosted) | Generate cited summary + next best action from retention playbook KB |
|7. Offline evaluation|Held-out set of player question pairs + groundedness/faithfulness scoring| Check retrieval precision/recall and confirm generated summaries stay grounded in retrieved messages (no hallucination) before the tool ships to agents|
| 8. Surface in CRM | CRM dashboard integration | Display summary + churn score for flagged players |
| 9. Feedback loop | Agent rating UI(for AI eval) + MLflow/Langfuse | Agents rate summary usefulness, extracted tags feed back to improve extraction model |

* **Note:** Precision, recall, and F1 from the churn model do not transfer directly to the RAG system because retrieval is a different task. Unlike churn prediction, retrieval does not naturally come with labeled relevant documents. We therefore need to create an evaluation dataset by having humans (or an LLM followed by human review) identify the relevant messages for a representative set of queries.
I would then evaluate the two stages separately. For retrieval, i would measure metrics such as Precision@k, Recall@k (or Hit Rate@k), and optionally MRR/nDCG against the annotated relevance set. For generation, i would evaluate faithfulness (whether the response is fully supported by the retrieved messages) and answer relevance/correctness.
Several of these metrics rely on LLM as a judge evaluation rather than being computed directly, which adds evaluation cost. Therefore, i would run the full evaluation suite periodically (e.g., on a fixed benchmark after changes to prompts, retrieval logic, or embeddings) rather than on every production query. For continuous production monitoring, I would rely on cheaper operational signals such as agent usefulness ratings, citation click-through, and retrieval latency.

* Now here with On-prem we can use LLM with open weight which can be specifically set for custom solution. Therefore company has two paths depending on the compliance posture:

    * **Hybrid:** Keep raw text, embeddings and vector DB on-prem and send only redacted retrieved snippets to a private hosted LLM for the generation step.
    * **Fully on-prem:** self-hosted open weight model (Llama / gpt-oss/ etc) via vLLM/TGI on internal GPUs. This depends on the experiments can result in lower quality but can be cheaper and good enough for the given problem.
    * **Track everything:** Tracings and cost per LLM call should be loged via Langfuse for better optimizations and tracking. MLflow for version trainc and experiment records.

Q.3:

5. Business Alignment
Imagine a situation where your model flags certain players as high-risk, but a business manager states in a meeting that the results do not align with their expectations from the field and doubts the model's accuracy. How would you approach this conversation and guide the communication with them?


A.3: 


Q.4:

6. Truth Verification
Using data and analytics, how would you check whether the business manager is right or if your model made an error? What experiment or metric would you propose to the business team to evaluate the situation based on facts rather than subjective feelings?


A.4: